# Install packages

In [1]:
!pip install fair-esm
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 3.1 MB/s  0:00:00m0:00:0100:010m


# Load packages and model

In [2]:
import torch
import esm
import os
from pathlib import Path
from Bio import SeqIO
from typing import List, Tuple, Optional
from tqdm import tqdm

# Load ESM-2 model
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()  # disables dropout for deterministic results

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /home/ec2-user/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /home/ec2-user/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt


ESM2(
  (embed_tokens): Embedding(33, 1280, padding_idx=1)
  (layers): ModuleList(
    (0-32): 33 x TransformerLayer(
      (self_attn): MultiheadAttention(
        (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (rot_emb): RotaryEmbedding()
      )
      (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (fc1): Linear(in_features=1280, out_features=5120, bias=True)
      (fc2): Linear(in_features=5120, out_features=1280, bias=True)
      (final_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    )
  )
  (contact_head): ContactPredictionHead(
    (regression): Linear(in_features=660, out_features=1, bias=True)
    (activation): Sigmoid()
  )
  (emb_layer_norm_after): LayerNorm((1280,), eps=1

# Generate all possible single substitutions

## Function

In [5]:
def generate_protein_mutations_fasta(sequence, protein_name, output_file):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    
    with open(output_file, 'w') as f:
        # Write original sequence
        f.write(f'>{protein_name}_WT\n{sequence}\n')
        
        # Generate all possible single mutations
        for i, original_aa in enumerate(sequence):
            for new_aa in amino_acids:
                if new_aa != original_aa:
                    mutated_sequence = sequence[:i] + new_aa + sequence[i+1:]
                    mutation_name = f'{original_aa}{i+1}{new_aa}'
                    f.write(f'>{protein_name}_{mutation_name}\n{mutated_sequence}\n')

## Run

In [6]:
sequence = 'MTEAQAIAKQLGGVKPDDEWLQAEIARLKGKSIVPLQQVKTLHDWLDGKRKARKSCRVVGESRTGKTVACDAYRYRHKPQQEAGRPPTVPVVYIRPHQKCGPKDLFKKITEYLKYRVTKGTVSDFRDRTIEVLKGCGVEMLIIDEADRLKPETFADVRDIAEDLGIAVVLVGTDRLDAVIKRDEQVLERFRAHLRFGKLSGEDFKNTVEMWEQMVLKLPVSSNLKSKEMLRILTSATEGYIGRLDEILREAAIRSLSRGLKKIDKAVLQEVAKEYK'
protein_name = 'ShCAST'
output_file = f'assets/{protein_name}_mutations.fasta'

generate_protein_mutations_fasta(sequence, protein_name, output_file)

# Generate ESM2 sequence embeddings

## Function

In [3]:
def generate_esm2_embeddings(
    fasta_file: str, 
    output_dir: str, 
    model_name: str = "esm2_t33_650M_UR50D",
    repr_layer: int = 33,
    batch_size: int = 1,
    device: Optional[str] = None
) -> None:
    """
    Generate ESM2 per-sequence embeddings from a FASTA file and save them.
    
    Args:
        fasta_file (str): Path to input FASTA file
        output_dir (str): Directory to save embeddings
        model_name (str): ESM2 model name (default: esm2_t33_650M_UR50D)
        repr_layer (int): Representation layer to extract (default: 33)
        batch_size (int): Batch size for processing (default: 1)
        device (str, optional): Device to use ('cpu' or 'cuda'). Auto-detects if None.
    
    Returns:
        None: Saves embeddings as .pt files in output_dir
    """
    
    # Create output directory if it doesn't exist
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Set device
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    
    print(f"Using device: {device}")
    
    # Load ESM2 model and alphabet
    model, alphabet = esm.pretrained.load_model_and_alphabet(model_name)
    model = model.to(device)
    model.eval()
    batch_converter = alphabet.get_batch_converter()
    
    # Read sequences from FASTA file
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        sequences.append((record.id, str(record.seq)))
    
    print(f"Loaded {len(sequences)} sequences from {fasta_file}")
    
    # Process sequences in batches with progress bar
    with tqdm(total=len(sequences), desc="Processing sequences", unit="seq") as pbar:
        for i in range(0, len(sequences), batch_size):
            batch_sequences = sequences[i:i + batch_size]
            batch_labels, batch_strs, batch_tokens = batch_converter(batch_sequences)
            batch_tokens = batch_tokens.to(device)
            batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)
            
            # Extract per-residue representations
            with torch.no_grad():
                results = model(batch_tokens, repr_layers=[repr_layer], return_contacts=False)
            
            token_representations = results["representations"][repr_layer]
            
            # Generate and save per-sequence representations immediately
            for j, (seq_id, _) in enumerate(batch_sequences):
                tokens_len = batch_lens[j]
                # NOTE: token 0 is always a beginning-of-sequence token, so the first residue is token 1.
                # token -1 is end-of-sequence token, so we exclude it with tokens_len - 1
                sequence_repr = token_representations[j, 1:tokens_len - 1].mean(0)
                
                # Save immediately to avoid memory accumulation
                embedding_file = os.path.join(output_dir, f"{seq_id}.pt")
                torch.save(sequence_repr, embedding_file)
                pbar.update(1)
    
    print(f"Successfully saved embeddings for {len(sequences)} sequences to {output_dir}")
    
    # Print embedding statistics from the first saved file (for reference)
    first_seq_id = sequences[0][0]
    first_embedding = torch.load(os.path.join(output_dir, f"{first_seq_id}.pt"))
    embedding_dim = first_embedding.shape[0]
    print(f"Embedding dimension: {embedding_dim}")

## Run

In [4]:
# Generate embeddings - creates individual .pt files
generate_esm2_embeddings("assets/ShCAST_mutations.fasta", "./assets/ShCAST_emb/")

Using device: cuda
Loaded 5245 sequences from assets/ShCAST_mutations.fasta


Processing sequences: 100%|██████████| 5245/5245 [11:46<00:00,  7.43seq/s]

Successfully saved embeddings for 5245 sequences to ./assets/ShCAST_emb/
Embedding dimension: 1280
